# Test — Reference Book Ingestion (standalone)

Exercises **only the reference KB pipeline**: PDF → chunk → **local GPU
embeddings** → persistent **ChromaDB** → similarity search. No LLM / OpenAI key
needed (ingestion + search are fully local).

Run this to confirm the two threat-modeling books index correctly and that the
GPU is used.

## Prerequisites

Run from the project root (`ai-threat-modeling-assistant/`):

```bash
pip install -r requirements.txt
# RTX 5090 (Blackwell) needs a CUDA 12.8+ torch build:
pip install torch --index-url https://download.pytorch.org/whl/cu128
```

The books must be in a `Reference/Books` folder (auto-discovered upward) or set
`REFERENCE_BOOKS_DIR`. First run downloads the embedding model (~130 MB).

In [ ]:
import logging
from dotenv import load_dotenv
import threat_model.references as R

load_dotenv()
logging.basicConfig(level="INFO", format="%(asctime)s | %(levelname)s | %(message)s")

books = R.resolve_books_dir()
print("Books dir       :", books)
print("Embedding model :", R.EMBEDDING_MODEL)
print("Device (auto)   :", R._device())
print("Chroma dir      :", R.CHROMA_DIR)
print("Collection      :", R.COLLECTION)
print("Chunk size/ovlp :", R.CHUNK_SIZE, "/", R.CHUNK_OVERLAP)
assert books, "No books found — set REFERENCE_BOOKS_DIR or add PDFs to Reference/Books"


## 1. Confirm GPU is available

The embedder auto-selects CUDA when present (your RTX 5090). If this prints
`False`, ingestion still works on CPU (slower).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Parse preview (fast, no embeddings)

Load + chunk each PDF without embedding, to sanity-check parsing and chunk counts.
Note: some "PDFs" are saved HTTP multipart uploads; the loader strips that wrapper
automatically.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=R.CHUNK_SIZE, chunk_overlap=R.CHUNK_OVERLAP)
for pdf in sorted(books.glob("*.pdf")):
    chunks = R._load_pdf_chunks(pdf, splitter)
    pages = max((c["metadata"]["page"] for c in chunks), default=0)
    print(f"{pdf.name}: {len(chunks)} chunks across ~{pages} text-pages")
    if chunks:
        print("   sample:", chunks[0]["id"], "|", chunks[0]["metadata"])


## 3. Ingest into ChromaDB (uses the GPU)

`ingest()` is **incremental** (per-file content hash → `data/reference_index_state.json`):
new/changed PDFs are embedded and stored, unchanged ones are skipped. Use
`force=True` to re-index everything.

In [ ]:
kb = R.ReferenceKB()
report = kb.ingest()          # PDF -> chunk -> GPU embed -> ChromaDB
print("Ingest report:", report)
print("Total chunks in collection:", kb.count())


## 4. Sample similarity searches

Confirm retrieval returns relevant passages with **source + page** metadata.

In [ ]:
queries = [
    "spoofing an API client to bypass authentication",
    "STRIDE threats for a data store",
    "mitigations for tampering with messages in transit",
]
for q in queries:
    print("=" * 90, "\nQUERY:", q)
    for h in kb.search(q, k=3):
        m = h["metadata"]
        print(f"  score={h['score']:.3f}  {m.get('source')} p.{m.get('page')}")
        print("   ", h["text"][:220].replace("\n", " "), "…")


## 5. Incremental re-ingest

Running again should **skip** the already-indexed files (no re-embedding).

In [ ]:
print("Re-running ingest (incremental):")
print(kb.ingest())
print("Total chunks:", kb.count())


## Notes

- This is the exact pipeline `service.generate_threat_model_report()` uses to
  ground generation: when the app is in **OpenAI mode**, it calls
  `references.retrieve_reference_context(description)` and injects the top passages
  into the prompt.
- Reset the index with `R.ReferenceKB().reset()` or `kb.ingest(force=True)`.
- CLI equivalent: `python ingest_references.py` (and `--stats`, `--search "..."`, `--force`).